# cll-cpm-inversion: quickstart

This notebook walks through the full pipeline end-to-end on a small
set of synthetic mask images. The same code works on real segmented
spheroid masks.

**Pipeline**

1. Generate a folder of mask files (you would normally already have
   these from your segmenter; here we create synthetic disks so the
   notebook is self-contained).
2. Extract the five morphology features from each mask.
3. Invert into CPM parameter posteriors.
4. Read the per-parameter identifiability flags and the posterior
   median + 90% interval.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image

from cll_cpm_inversion import (
    OPERATIONAL_FEATURES,
    PARAMS,
    extract_features_from_mask,
    features_from_folder,
    infer_from_features,
    infer_from_masks,
    load_identifiability,
)

## 1. Create a synthetic mask folder

Replace this step with your own segmented spheroid masks. The
package accepts `.jpg`, `.png`, `.tif`, and `.tiff` files. Pixels
with value > 0 are treated as spheroid.

In [2]:
WORK = Path("/tmp/cll_quickstart")
WORK.mkdir(exist_ok=True)

def disk(radius: int, size: int = 200) -> np.ndarray:
    y, x = np.ogrid[:size, :size]
    cy = cx = size // 2
    return ((y - cy) ** 2 + (x - cx) ** 2 <= radius ** 2).astype(np.uint8) * 255

# Layout: one subfolder per spheroid, frames inside
for spheroid_id, radii in [
    ("W001_unstim", [20, 22, 24, 26]),
    ("W002_unstim", [22, 25, 28, 30]),
    ("W003_unstim", [18, 20, 21, 22]),
    ("W004_stim",   [30, 35, 40, 45]),
    ("W005_stim",   [28, 33, 38, 42]),
]:
    sd = WORK / spheroid_id
    sd.mkdir(exist_ok=True)
    for frame, r in enumerate(radii):
        Image.fromarray(disk(r)).save(sd / f"frame_{frame:02d}.tif")

print("Created", WORK, "with", len(list(WORK.iterdir())), "trajectory folders.")

Created /tmp/cll_quickstart with 5 trajectory folders.


## 2. Extract features

`features_from_folder` walks the folder. With one subfolder per
spheroid (the layout above), it averages the per-frame features into
one representative vector per spheroid. Set `aggregate=False` to keep
one row per (spheroid, frame).

In [3]:
features_df = features_from_folder(WORK)
features_df.round(3)

,spheroid_id,total_area,equivalent_diameter,solidity,perimeter,circularity,n_frames
0,W001_unstim,1672.0,45.925,0.970,151.196,0.910,4
1,W002_unstim,2188.0,52.434,0.974,173.095,0.905,4
2,W003_unstim,1289.0,40.402,0.966,133.296,0.907,4
3,W004_stim,4515.0,74.989,0.983,247.279,0.908,4
4,W005_stim,3975.0,70.361,0.980,232.208,0.907,4


## 3. Run the inversion

The simplest call: `infer_from_masks(folder)` runs steps 2 and 3
together. Here we already have the features, so we use
`infer_from_features` directly.

In [4]:
posterior = infer_from_features(features_df, k=20)
posterior.round(3).head(20)

,spheroid_id,parameter,median,q05,q95,q25,q75,n_matches,loo_r2,identifiability
0,W001_unstim,width,6.500,4.000,10.150,5.000,10.000,20,0.466,weakly identifiable
1,W001_unstim,temp,42.492,12.750,69.839,20.963,65.020,20,0.134,unidentifiable
2,W001_unstim,lambda,0.346,0.152,2.470,0.218,0.784,20,0.042,unidentifiable
3,W001_unstim,contact,37.367,22.993,47.225,32.199,43.301,20,0.310,weakly identifiable
4,W001_unstim,cm_adhesion,35.453,19.184,46.536,26.074,39.855,20,0.613,weakly identifiable
5,W001_unstim,contact_no,6.000,2.000,8.000,4.000,8.000,20,0.127,unidentifiable
6,W001_unstim,neighbor,4.000,1.000,6.000,2.000,5.000,20,-0.090,unidentifiable
7,W002_unstim,width,6.500,4.000,10.150,5.000,10.000,20,0.466,weakly identifiable
8,W002_unstim,temp,42.492,12.750,69.839,20.963,65.020,20,0.134,unidentifiable
9,W002_unstim,lambda,0.346,0.152,2.470,0.218,0.784,20,0.042,unidentifiable


## 4. Read the identifiability flags

The library's leave-one-out R^2 tells us how recoverable each
parameter is from 2D morphology *at all*. This is a property of the
library, not of your spheroid - it bounds how seriously to read each
row of the posterior.

In [5]:
load_identifiability().round(3)

,param,R2,pearson,n
0,width,0.466,0.686,1105
1,temp,0.134,0.411,1105
2,contact,0.310,0.572,1105
3,neighbor,-0.090,0.110,1105
4,contact_no,0.127,0.443,1105
5,lambda,0.042,0.357,1105
6,cm_adhesion,0.613,0.789,1105


Three of the seven parameters cross the 0.3 threshold and are flagged
`weakly identifiable`: width, contact ($J_{cc}$), and cm_adhesion
($J_{cm}$). The other four are `unidentifiable` from a single 2D
spheroid snapshot and their posterior values should be read as the
prior expectation, not as point estimates.

## 5. Pivot for a clean report

Most users will want a wide table: one row per spheroid, one column
per identifiable parameter.

In [6]:
identifiable = posterior[posterior["identifiability"] != "unidentifiable"]
wide_median = identifiable.pivot(index="spheroid_id",
                                 columns="parameter",
                                 values="median").round(2)
wide_median

parameter,cm_adhesion,contact,width
spheroid_id,,,
W001_unstim,35.45,37.37,6.5
W002_unstim,35.45,37.37,6.5
W003_unstim,35.45,37.37,6.5
W004_stim,35.45,37.37,6.5
W005_stim,35.45,37.37,6.5


## 6. Comparing conditions

When you have multiple conditions (stim vs unstim, drug vs vehicle,
patient A vs patient B), the *defensible read* is the cross-condition
*shift* on the identifiable axes, not absolute parameter values
(~90% of real CLL spheroids extrapolate the bundled library, see
`docs/METHODS.md` section 6).

In [7]:
features_df["condition"] = features_df["spheroid_id"].str.split("_").str[-1]
features_df.groupby("condition")[OPERATIONAL_FEATURES].mean().round(2)

,total_area,equivalent_diameter,solidity,perimeter,circularity
condition,,,,,
stim,4245.00,72.68,0.98,239.74,0.91
unstim,1716.33,46.25,0.97,152.53,0.91


In [8]:
posterior["condition"] = posterior["spheroid_id"].str.split("_").str[-1]
by_condition = (posterior[posterior["identifiability"] != "unidentifiable"]
                .groupby(["condition", "parameter"])["median"]
                .median().unstack("parameter").round(2))
by_condition

parameter,cm_adhesion,contact,width
condition,,,
stim,35.45,37.37,6.5
unstim,35.45,37.37,6.5


## 7. CLI equivalent

Everything above can be done from the shell:

```bash
cll-invert /tmp/cll_quickstart --out /tmp/posteriors.csv
```

The CLI writes the same long posterior summary CSV and prints the
pivoted table of identifiable parameter medians.